# Entraînement — Mini-DDSM (sous-ensemble JPEG-8)

> ### ⚠️ Modèle entraîné, pas un modèle clinique
>
> Ce notebook entraîne un classifieur bénin / malin sur **Mini-DDSM**, en
> remplacement du modèle mini-MIAS intérimaire (115 images lésionnelles) qui a
> servi à valider la mécanique. Mini-DDSM apporte un ordre de grandeur de plus :
> environ 1 350 images exploitables, réparties sur plusieurs centaines de cas.
>
> Plus de données ne vaut pas validation clinique. Le checkpoint produit ici
> sort avec `clinically_validated: False`, comme le précédent, et le restera
> tant qu'aucune revue humaine documentée n'aura eu lieu — voir
> `DEFAULT_CLINICALLY_VALIDATED` dans `app/ai/inference/loader.py`. Aucune
> métrique de ce notebook n'autorise à changer ce drapeau.

## Ce que le notebook garantit

| Point | Garantie |
|-------|----------|
| Prétraitement | Importé de `app.ai.preprocessing`, pas réimplémenté ici |
| Ordre des classes | Importé de `app.ai.CLASS_NAMES`, jamais écrit en dur |
| Découpage | Par **cas / patiente**, jamais par image (4 clichés par cas) |
| Étiquetage | Asymétrique et assumé : `malignant` seulement sur lésion annotée (étape 2 bis) |
| Effectifs | **Comptés sur les fichiers réels**, aucun chiffre supposé |
| Checkpoint | Relu par `app.ai.inference.loader.load_checkpoint` avant la fin du notebook |

## Données attendues

```
<MINIDDSM_ROOT>/
├── Benign/<cas>/<PRÉFIXE>_<numéro>_<étude>.<CÔTÉ>_<INCIDENCE>.jpg
├── Cancer/<cas>/…
│   └── …_Mask.jpg        # masque de lésion : dit quels clichés en portent une
└── Normal/<cas>/…        # comptés puis écartés, voir l'étape 2
```

Pas de fichier d'annotations : contrairement à mini-MIAS et son `Info.txt`,
**l'étiquette vient du dossier de premier niveau**, le regroupement par patiente
vient du **nom de fichier** (`C_0251_1.LEFT_MLO` → cas `C_0251`), et la présence
d'une lésion vient des **masques**. Les trois hypothèses sont vérifiées aux
étapes 1 et 2, pas supposées.


## Configuration

Les valeurs se surchargent par variables d'environnement, ce qui permet de
rejouer le notebook en local sur un échantillon (`BREASTAI_SMOKE=1`) avant de le
lancer sur Colab.


In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules


def _default_repo_root() -> Path:
    """Remonte jusqu'au dossier contenant `backend/app/ai`."""
    here = Path.cwd()
    for candidate in (here, *here.parents):
        if (candidate / "backend" / "app" / "ai").is_dir():
            return candidate
    return Path("/content/BreastAi") if IN_COLAB else here


# Dépôt : sur Colab il est cloné à l'étape suivante, en local il est déjà là.
REPO_ROOT = Path(os.environ.get("BREASTAI_REPO_ROOT", _default_repo_root()))
REPO_URL = os.environ.get("BREASTAI_REPO_URL", "https://github.com/MAHAMAT767/BreastAi.git")

# Dataset Mini-DDSM. Le chemin Drive exact se règle sans toucher au notebook.
MINIDDSM_ROOT = Path(
    os.environ.get(
        "BREASTAI_MINIDDSM_ROOT",
        "/content/drive/MyDrive/datasets/MINI-DDSM-Complete-JPEG-8"
        if IN_COLAB
        else r"C:\Dev\datasets\MINI-DDSM-Complete-JPEG-8",
    )
)

# Provenance du jeu de données, reportée telle quelle dans la fiche modèle : à
# corriger si le dataset a été obtenu ailleurs que par cette source.
DATASET_NAME = os.environ.get("BREASTAI_DATASET_NAME", "Mini-DDSM (JPEG-8)")
DATASET_SOURCE = os.environ.get(
    "BREASTAI_DATASET_SOURCE", "https://www.kaggle.com/datasets/cheddad/miniddsm2"
)

# Sorties : checkpoint et fiche modèle dans `models/`, dossier exclu du dépôt
# par .gitignore — des mammographies et des poids n'ont rien à faire dans un
# commit. Aucun cache d'images n'est nécessaire ici : les fichiers sont déjà
# dans un format que le pipeline sait lire (voir l'étape 4).
OUTPUT_DIR = Path(os.environ.get("BREASTAI_OUTPUT_DIR", REPO_ROOT / "models"))

# Mode vérification : quelques cas, une époque. Sert à valider la mécanique du
# notebook, pas à produire un modèle.
SMOKE_TEST = os.environ.get("BREASTAI_SMOKE", "0") == "1"

SEED = 20260815
ARCHITECTURE = "efficientnet_b0"
MODEL_VERSION = os.environ.get("BREASTAI_MODEL_VERSION", "efficientnet_b0-miniddsm-v1")

BATCH_SIZE = 4 if SMOKE_TEST else 16
EPOCHS = 1 if SMOKE_TEST else 30
EARLY_STOPPING_PATIENCE = 1 if SMOKE_TEST else 8
LR_HEAD = 1e-3
LR_BACKBONE = 1e-4
WEIGHT_DECAY = 1e-4

# Sensibilité visée pour le choix du seuil : en dépistage, un faux négatif coûte
# bien plus cher qu'un faux positif.
TARGET_SENSITIVITY = 0.90

# Mode vérification : on échantillonne des **cas entiers**, pas des images. Tirer
# des images isolées casserait précisément ce que le notebook doit vérifier — le
# découpage par patiente.
SMOKE_MAX_PATIENTS_PER_CLASS = int(os.environ.get("BREASTAI_SMOKE_PATIENTS", "6"))

for name, value in [
    ("REPO_ROOT", REPO_ROOT),
    ("MINIDDSM_ROOT", MINIDDSM_ROOT),
    ("OUTPUT_DIR", OUTPUT_DIR),
    ("SMOKE_TEST", SMOKE_TEST),
]:
    print(f"{name:14} = {value}")


### Environnement Colab

Sur Colab : montage de Drive, clonage du dépôt, installation des dépendances
d'imagerie. En local, cette cellule ne fait rien.


In [ ]:
if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")

    if not (REPO_ROOT / "backend" / "app" / "ai").is_dir():
        # `--depth 1` : seul l'état courant du code nous intéresse.
        os.system(f"git clone --depth 1 {REPO_URL} {REPO_ROOT}")

    # torch, torchvision et numpy sont déjà fournis par Colab. Restent les
    # paquets d'imagerie d'app.ai.preprocessing, et pydantic-settings :
    # `app.ai.inference.loader` importe `app.config` pour connaître MODEL_PATH,
    # ce qui tire toute la configuration de l'application. `NoDecode` exige la
    # version 2.7 ou plus récente.
    os.system(
        "pip install --quiet opencv-python-headless pydicom scikit-learn "
        "'pydantic-settings>=2.7'"
    )

if not (REPO_ROOT / "backend" / "app" / "ai").is_dir():
    raise FileNotFoundError(
        f"{REPO_ROOT} ne contient pas backend/app/ai. Le prétraitement doit être "
        "importé du dépôt et non recopié ici : sans lui, le modèle serait entraîné "
        "sur des images préparées autrement qu'à l'inférence."
    )

# `backend/` d'abord : c'est là que vit le paquet `app`.
sys.path.insert(0, str(REPO_ROOT / "backend"))


### Imports et vérification du contrat

Tout ce qui doit rester identique entre l'entraînement et l'inférence est
**importé**, jamais recopié : taille d'entrée, ordre des classes, chaîne de
prétraitement, fabrique du modèle.


In [ ]:
import json
import random
import re
import subprocess
from collections import Counter, defaultdict
from dataclasses import dataclass
from datetime import datetime, timezone

import cv2
import numpy as np
import torch
from sklearn.metrics import (
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from torch import nn
from torch.utils.data import DataLoader, Dataset

from app.ai import CLASS_LABELS_FR, CLASS_NAMES, IMAGE_SIZE
from app.ai.inference.loader import build_model, load_checkpoint
from app.ai.inference.predictor import MALIGNANT_INDEX, Predictor
from app.ai.preprocessing.loaders import ALLOWED_EXTENSIONS, ImageFormat, detect_format
from app.ai.preprocessing.pipeline import PREPROCESSING_VERSION, preprocess_for_inference
from app.ai.preprocessing.transforms import normalize

# Le contrat de sortie du modèle est figé côté code : le notebook s'y conforme
# au lieu de le redéfinir.
assert CLASS_NAMES == ("benign", "malignant"), CLASS_NAMES
assert CLASS_NAMES[MALIGNANT_INDEX] == "malignant"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("classes            :", CLASS_NAMES)
print("taille d'entrée    :", IMAGE_SIZE)
print("prétraitement      :", PREPROCESSING_VERSION)
print("périphérique       :", DEVICE)


## Étape 1 — Inventaire des fichiers

Mini-DDSM n'a pas d'`Info.txt` : l'arborescence **est** l'annotation. Deux
informations en sont tirées, et chacune est vérifiée plutôt que supposée.

**L'étiquette vient du dossier de premier niveau** : `Benign` → `benign`,
`Cancer` → `malignant`, `Normal` → écarté (étape 2). L'absence de l'un des trois
fait échouer la cellule. Un dossier supplémentaire — certaines distributions en
ajoutent un — est ignoré mais signalé, avec le nombre d'images qu'il contient :
rien n'en est tiré, et cela doit rester visible.

**Le cas patiente vient du nom de fichier** : `C_0251_1.LEFT_MLO` → cas
`C_0251`. Le chiffre d'étude (`_1`) est volontairement exclu de la clé : deux
études d'une même patiente restent une même patiente. Tout fichier dont le nom
ne suit pas cette forme est ignoré et signalé — fichiers annexes, vignettes,
tout ce qui n'est pas une mammographie et fausserait l'entraînement.

Les masques de lésion (`…_Mask.jpg`, parfois numérotés `…_Mask_1.jpg`) font
exception : ils ne sont pas des images d'entraînement, mais ils disent **quels
clichés portent effectivement une lésion**. Ils sont donc indexés, pas
seulement écartés — l'étape 2 s'en sert pour l'étiquetage des cas `Cancer`.

Un doublon d'extension (`.jpg` et `.png` pour le même cliché) est résolu en
faveur du JPEG-8, le sous-ensemble annoncé.


In [ ]:
LABEL_NORMAL = "normal"

#: Étiquette portée par chaque dossier de premier niveau. `Normal` est présent
#: ici pour être compté, pas pour être appris (voir l'étape 2).
LABEL_BY_FOLDER = {
    "benign": "benign",
    "cancer": "malignant",
    "normal": LABEL_NORMAL,
}

#: Forme canonique d'un nom de cliché DDSM : préfixe de volume, numéro de cas,
#: numéro d'étude, côté, incidence. Tout ce qui n'y correspond pas — vignettes,
#: fichiers annexes — est écarté et compté.
FILENAME_PATTERN = re.compile(
    r"^(?P<volume>[A-Z])_(?P<case>\d+)_(?P<study>\d+)"
    r"\.(?P<side>LEFT|RIGHT)_(?P<view>CC|MLO)$",
    re.IGNORECASE,
)

#: Masque de lésion accompagnant un cliché. Le suffixe numéroté couvre les
#: clichés porteurs de plusieurs lésions (`…_Mask_1.jpg`, `…_Mask_2.jpg`).
MASK_PATTERN = re.compile(
    r"^(?P<volume>[A-Z])_(?P<case>\d+)_(?P<study>\d+)"
    r"\.(?P<side>LEFT|RIGHT)_(?P<view>CC|MLO)_Mask(?:_\d+)?$",
    re.IGNORECASE,
)

#: Ordre de préférence quand le même cliché existe sous plusieurs extensions.
EXTENSION_PRIORITY = (".jpg", ".jpeg", ".png")


def cliche_key(matched: re.Match[str]) -> str:
    """Identifiant d'un cliché, partagé par l'image et ses masques.

    C'est ce qui permet de rapprocher `C_0251_1.LEFT_CC.jpg` de
    `C_0251_1.LEFT_CC_Mask.jpg` sans dépendre du numéro d'étude.
    """
    return (
        f"{matched['volume'].upper()}_{matched['case']}"
        f".{matched['side'].upper()}_{matched['view'].upper()}"
    )


@dataclass(frozen=True)
class Mammogram:
    """Un cliché Mini-DDSM et ce qu'on sait de lui."""

    path: Path
    label: str
    patient: str
    side: str
    view: str
    folder: str

    @property
    def key(self) -> str:
        """Identifiant du cliché, indépendant de l'extension."""
        return f"{self.patient}.{self.side}_{self.view}"


def inventory(root: Path) -> tuple[list[Mammogram], Counter, Counter]:
    """Parcourt l'arborescence.

    Retourne (clichés, masques par cliché, fichiers écartés par motif).
    """
    if not root.is_dir():
        raise FileNotFoundError(
            f"{root} introuvable. Renseigner BREASTAI_MINIDDSM_ROOT avec le "
            "dossier contenant Benign/, Cancer/ et Normal/."
        )

    present = {
        entry.name.lower(): entry for entry in sorted(root.iterdir()) if entry.is_dir()
    }
    unexpected = sorted(set(present) - set(LABEL_BY_FOLDER))
    missing = sorted(set(LABEL_BY_FOLDER) - set(present))
    if missing:
        raise FileNotFoundError(
            f"{root} : dossiers de classe absents {missing}. Attendu : "
            f"{sorted(LABEL_BY_FOLDER)}."
        )
    if unexpected:
        # Certaines distributions Mini-DDSM ajoutent un dossier annexe à côté des
        # trois classes (`Data-MoreThanTwoMasks`). Aucune image n'en est tirée :
        # s'arrêter là n'apporterait rien, mais le passer sous silence cacherait
        # un dossier plein de clichés. D'où le décompte.
        print("⚠️ dossiers de premier niveau ignorés (aucune étiquette associée) :")
        for name in unexpected:
            ignored = sum(
                1
                for path in present[name].rglob("*")
                if path.is_file() and path.suffix.lower() in EXTENSION_PRIORITY
            )
            print(f"   {present[name].name:30} {ignored:5} image(s) écartée(s)")
        print()

    # Un même cliché peut exister en .jpg et en .png : on garde le meilleur selon
    # EXTENSION_PRIORITY, sans quoi il compterait deux fois dans les effectifs.
    best_by_key: dict[str, Mammogram] = {}
    masks: Counter = Counter()
    skipped: Counter = Counter()

    for folder_name, label in LABEL_BY_FOLDER.items():
        for path in sorted(present[folder_name].rglob("*")):
            if not path.is_file():
                continue

            suffix = path.suffix.lower()
            if suffix not in EXTENSION_PRIORITY:
                skipped[f"extension {suffix or '(aucune)'}"] += 1
                continue

            # Les masques sont indexés avant d'être écartés comme images : c'est
            # d'eux que vient l'information « ce cliché porte une lésion ».
            mask = MASK_PATTERN.match(path.stem)
            if mask is not None:
                masks[cliche_key(mask)] += 1
                skipped["masque de lésion (indexé)"] += 1
                continue

            matched = FILENAME_PATTERN.match(path.stem)
            if matched is None:
                # Un masque au nom inattendu compte double : il n'entre pas dans
                # l'index, donc son cliché passera pour dépourvu de lésion.
                reason = (
                    "masque au nom hors forme"
                    if "mask" in path.stem.lower()
                    else "nom hors forme"
                )
                skipped[reason] += 1
                continue

            image = Mammogram(
                path=path,
                label=label,
                patient=f"{matched['volume'].upper()}_{matched['case']}",
                side=matched["side"].upper(),
                view=matched["view"].upper(),
                folder=present[folder_name].name,
            )

            previous = best_by_key.get(image.key)
            if previous is None:
                best_by_key[image.key] = image
            elif EXTENSION_PRIORITY.index(suffix) < EXTENSION_PRIORITY.index(
                previous.path.suffix.lower()
            ):
                best_by_key[image.key] = image
                skipped["doublon d'extension"] += 1
            else:
                skipped["doublon d'extension"] += 1

    return sorted(best_by_key.values(), key=lambda m: m.key), masks, skipped


images, masks, skipped = inventory(MINIDDSM_ROOT)
if not images:
    raise FileNotFoundError(f"{MINIDDSM_ROOT} : aucun cliché exploitable trouvé.")

counts_by_label = Counter(image.label for image in images)
patients_by_label = {
    label: {image.patient for image in images if image.label == label}
    for label in counts_by_label
}

print(f"{MINIDDSM_ROOT}\n")
print(f"{len(images)} clichés retenus, {len({i.patient for i in images})} cas\n")
for folder_name, label in LABEL_BY_FOLDER.items():
    print(
        f"  {folder_name:8} → {label:10} {counts_by_label.get(label, 0):5} clichés"
        f"  {len(patients_by_label.get(label, ())):4} cas"
    )

print(f"\n{sum(masks.values())} masques de lésion sur {len(masks)} clichés distincts")

print("\nfichiers non retenus comme images :")
for reason, count in sorted(skipped.items()) or [("aucun", 0)]:
    print(f"  {reason:28} {count:5}")

# Les extensions rencontrées doivent être celles que l'API accepte à l'import :
# entraîner sur un format que le pipeline refuserait n'aurait pas de sens.
extensions = {image.path.suffix.lower() for image in images}
assert extensions <= ALLOWED_EXTENSIONS, extensions
print("\nextensions rencontrées :", sorted(extensions))


### Vérification des deux hypothèses de nommage

Le découpage par patiente ne vaut que si la clé tirée du nom de fichier désigne
bien un cas, et un seul. Deux contrôles :

1. **Bijection cas ↔ dossier** : chaque clé `C_0251` correspond à exactement un
   dossier de cas, et chaque dossier de cas à une seule clé. Si l'un ou l'autre
   échoue, la lecture du nom de fichier est fausse et le découpage laisserait
   fuiter une patiente entre entraînement et validation — la cellule s'arrête.
2. **Un cas, une classe** : un cas dont les clichés seraient répartis entre
   `Benign` et `Cancer` serait signalé. L'étiquette restant portée par l'image,
   ce n'est pas bloquant, mais cela doit se voir.


In [ ]:
def check_patient_keys(images: list[Mammogram]) -> None:
    """Vérifie que la clé tirée du nom de fichier désigne bien un dossier de cas."""
    folders_by_patient: dict[str, set[tuple[str, str]]] = defaultdict(set)
    patients_by_folder: dict[tuple[str, str], set[str]] = defaultdict(set)

    for image in images:
        # Le dossier de classe fait partie de la clé de dossier : deux cas
        # homonymes dans Benign/ et Cancer/ ne sont pas le même dossier.
        folder = (image.folder, image.path.parent.name)
        folders_by_patient[image.patient].add(folder)
        patients_by_folder[folder].add(image.patient)

    split_cases = {p: f for p, f in folders_by_patient.items() if len(f) > 1}
    merged_folders = {f: p for f, p in patients_by_folder.items() if len(p) > 1}

    if split_cases or merged_folders:
        raise ValueError(
            "le nom de fichier ne désigne pas le dossier de cas : "
            f"{len(split_cases)} cas éclatés sur plusieurs dossiers "
            f"(ex. {list(split_cases.items())[:3]}), "
            f"{len(merged_folders)} dossiers regroupant plusieurs cas "
            f"(ex. {list(merged_folders.items())[:3]}). "
            "Le découpage par patiente serait faux : corriger la lecture du nom "
            "avant d'entraîner quoi que ce soit."
        )

    print(
        f"bijection vérifiée : {len(folders_by_patient)} cas ↔ "
        f"{len(patients_by_folder)} dossiers"
    )


def check_single_label(images: list[Mammogram]) -> None:
    """Signale les cas dont les clichés portent plusieurs étiquettes."""
    labels_by_patient: dict[str, set[str]] = defaultdict(set)
    for image in images:
        labels_by_patient[image.patient].add(image.label)

    mixed = sorted(p for p, labels in labels_by_patient.items() if len(labels) > 1)
    if mixed:
        print(
            f"⚠️ {len(mixed)} cas portent plusieurs étiquettes (ex. {mixed[:5]}). "
            "L'étiquette reste celle de l'image ; le regroupement par cas empêche "
            "toujours la fuite entre découpages."
        )
    else:
        print("chaque cas porte une seule étiquette")


check_patient_keys(images)
check_single_label(images)

# Cohérence de l'index de masques : un masque sans cliché correspondant signale
# une clé mal lue, et donc une restriction (étape 2) qui écarterait à tort.
orphans = sorted(set(masks) - {image.key for image in images})
if orphans:
    print(
        f"\n⚠️ {len(orphans)} masque(s) sans cliché correspondant "
        f"(ex. {orphans[:3]}) : vérifier MASK_PATTERN avant de se fier à l'index."
    )

# Répartition des clichés par cas : elle situe le poids d'une patiente dans le
# découpage. DDSM en attend quatre (deux côtés × deux incidences), mais les cas
# incomplets existent.
per_patient = Counter(Counter(i.patient for i in images).values())
print("\nclichés par cas :")
for number, cases in sorted(per_patient.items()):
    print(f"  {number} cliché(s) : {cases:4} cas")


### `DataWMask.xlsx` : à examiner avant de s'en servir

Certaines distributions Mini-DDSM livrent un `DataWMask.xlsx` à la racine, à
côté de `BoundaryMask.png`. S'il contient une table cas → clichés porteurs de
lésion, ce serait une source plus sûre que la lecture des noms de fichiers
`_Mask` : une convention de nommage se lit toujours un peu au jugé, une colonne
non.

Cette cellule ne fait que **montrer** ce que contient le fichier — colonnes,
dimensions, premières lignes. Elle ne l'utilise pas : brancher l'étiquetage sur
un schéma de colonnes non vérifié reviendrait à déplacer le pari, pas à le
supprimer. Une fois la structure connue, l'index de masques de l'étape 1 pourra
être remplacé en connaissance de cause.


In [ ]:
MASK_TABLE_PATH = MINIDDSM_ROOT / "DataWMask.xlsx"

if not MASK_TABLE_PATH.is_file():
    print(f"{MASK_TABLE_PATH.name} absent : l'index de masques vient des noms de fichiers.")
else:
    try:
        import pandas as pd

        table = pd.read_excel(MASK_TABLE_PATH)
    except ImportError as exc:
        # pandas et openpyxl sont fournis par Colab ; en local ils peuvent manquer.
        print(f"lecture impossible ({exc}) : pip install pandas openpyxl pour inspecter.")
    except Exception as exc:
        print(f"{MASK_TABLE_PATH.name} illisible ({exc}).")
    else:
        print(f"{MASK_TABLE_PATH.name} : {table.shape[0]} lignes, {table.shape[1]} colonnes\n")
        print("colonnes :", list(table.columns), "\n")
        print(table.head(10).to_string())
        print(
            "\nℹ️ Table affichée à titre d'information : l'étiquetage de l'étape 2 "
            "reste fondé sur les masques indexés à l'étape 1."
        )


## Étape 2 — Pourquoi les images normales sont écartées

Le modèle déployé a **deux** sorties, dans un ordre figé par le code :
`app.ai.CLASS_NAMES == ("benign", "malignant")`. `predictor.py` lit la
probabilité de l'index 1 comme « probabilité de malignité », `loader.py` refuse
au chargement tout checkpoint dont l'ordre des classes diffère, et l'API expose
`prediction ∈ {benign, malignant}`.

Ajouter une classe `normal` changerait ce contrat de bout en bout : sortie du
réseau, seuil, schémas d'API, base de données, rapports PDF. Ce n'est pas une
décision qui se prend dans un notebook d'entraînement — et c'est le même
traitement que celui appliqué aux 207 clichés normaux de mini-MIAS.

Le dossier `Normal` est donc **inventorié et compté** — il fait partie du
rapport ci-dessus — mais **exclu de l'entraînement**. « Normal » n'est pas
« bénin » : un sein sans lésion n'est pas un sein porteur d'une lésion bénigne,
et les confondre apprendrait au modèle une définition fausse du bénin.


## Étape 2 bis — Un étiquetage volontairement asymétrique

Mini-DDSM range les clichés par **cas**, pas par lésion. Un cas `Cancer` compte
jusqu'à quatre clichés : les deux incidences du sein porteur de la tumeur, et
les deux du sein controlatéral, souvent sain. Étiqueter les quatre `malignant`
— ce que ferait la lecture naïve de l'arborescence — apprendrait au modèle
qu'un tissu sain est un cancer.

Les deux classes ne sont donc **pas traitées de la même manière** :

| Dossier | Clichés retenus | Étiquette |
|---------|-----------------|-----------|
| `Cancer` | seulement ceux portant un **masque de lésion** | `malignant` |
| `Benign` | **tous** les clichés du cas | `benign` |
| `Normal` | aucun | — |

L'asymétrie suit celle du coût de l'erreur. Poser `malignant` sur une image
saine est une association fausse dans le sens le plus dangereux : le modèle
apprend à voir du cancer là où il n'y en a pas, la spécificité s'effondre, et
le signal que le rappel doit capter se brouille. Poser `benign` sur le cliché
sain d'un cas bénin reste, du point de vue du triage, une étiquette **juste** :
bénin et normal disent tous les deux « pas de cancer », et c'est exactement ce
que la sortie `benign` signifie dans un contrat à deux classes.

Autrement dit, on n'accepte du bruit d'étiquetage que là où il ne change pas la
réponse clinique. C'est aussi une différence de fond avec mini-MIAS, dont
l'`Info.txt` annotait chaque cliché individuellement et rendait la question sans
objet.

Conséquence à garder en tête : un cas `Cancer` dont aucun cliché ne porte de
masque disparaît entièrement de l'entraînement, et la classe maligne perd des
images. La cellule ci-dessous chiffre exactement ce qui est écarté.


In [ ]:
def restrict_malignant_to_lesion(
    images: list[Mammogram], masks: Counter
) -> tuple[list[Mammogram], dict[str, int]]:
    """Ne garde des cas `Cancer` que les clichés portant un masque de lésion.

    Les clichés bénins ne sont pas filtrés : l'étiquette `benign` reste juste sur
    un sein sain, alors que `malignant` ne l'est pas. Voir l'étape 2 bis.
    """
    def has_lesion(image: Mammogram) -> bool:
        return image.label != "malignant" or image.key in masks

    kept = [image for image in images if has_lesion(image)]
    dropped = [image for image in images if not has_lesion(image)]

    malignant_cases = {i.patient for i in images if i.label == "malignant"}
    kept_cases = {i.patient for i in kept if i.label == "malignant"}

    report = {
        "malignant_kept": sum(1 for i in kept if i.label == "malignant"),
        "malignant_dropped": len(dropped),
        "cases_lost": len(malignant_cases - kept_cases),
        # Des masques existent aussi sur des clichés bénins : ils ne servent pas
        # au filtrage, mais leur nombre dit si l'index couvre les deux classes.
        "benign_with_mask": sum(
            1 for i in kept if i.label == "benign" and i.key in masks
        ),
    }
    return kept, report


trainable = [image for image in images if image.label != LABEL_NORMAL]
trainable, mask_report = restrict_malignant_to_lesion(trainable, masks)

print("restriction des clichés malins aux porteurs de lésion :")
print(f"  retenus (masque présent)      {mask_report['malignant_kept']:5}")
print(f"  écartés (aucun masque)        {mask_report['malignant_dropped']:5}")
print(f"  cas Cancer entièrement perdus {mask_report['cases_lost']:5}")
print(f"  clichés bénins masqués (info) {mask_report['benign_with_mask']:5}\n")

if mask_report["malignant_kept"] == 0:
    raise ValueError(
        "aucun cliché Cancer ne porte de masque de lésion : cette distribution "
        "Mini-DDSM ne contient pas les masques, ou MASK_PATTERN ne reconnaît pas "
        "leur nommage. Vérifier le rapport de l'étape 1 avant de continuer — "
        "retomber sur un étiquetage par cas collerait `malignant` sur des seins "
        "sains, ce que l'étape 2 bis refuse explicitement."
    )

if SMOKE_TEST:
    # Échantillon de **cas entiers** et équilibré : la mécanique doit être testée
    # sur les deux classes, sans casser le regroupement par patiente.
    keep: set[str] = set()
    for label in CLASS_NAMES:
        patients = sorted({i.patient for i in trainable if i.label == label})
        keep.update(patients[:SMOKE_MAX_PATIENTS_PER_CLASS])
    kept = [image for image in trainable if image.patient in keep]
    print(
        f"MODE VÉRIFICATION : {len(kept)} clichés retenus sur {len(trainable)}, "
        f"{len(keep)} cas.\n"
    )
    trainable = kept

counts = {name: sum(1 for i in trainable if i.label == name) for name in CLASS_NAMES}
normal_count = counts_by_label.get(LABEL_NORMAL, 0)

print(f"clichés d'entraînement : {len(trainable)}")
for name in CLASS_NAMES:
    print(f"  {CLASS_LABELS_FR[name]:8} ({name:9}) {counts[name]:5}")
print(f"  {'écartés':8} ({LABEL_NORMAL:9}) {normal_count:5}")

if min(counts.values()) == 0:
    raise ValueError(
        "une des deux classes est vide : rien à apprendre. Vérifier "
        "BREASTAI_MINIDDSM_ROOT et le contenu des dossiers Benign/ et Cancer/."
    )


## Étape 3 — Découpage par patiente

Un cas Mini-DDSM porte jusqu'à quatre clichés : sein gauche et droit, incidences
CC et MLO. Répartir les images au hasard mettrait plusieurs clichés d'une même
patiente — voire les deux incidences du **même sein** — de part et d'autre du
découpage. Le modèle reverrait en validation un tissu déjà vu en entraînement,
et les métriques seraient flatteuses et fausses.

Le regroupement se fait donc sur la clé de cas vérifiée à l'étape 1. C'est le
même principe que pour mini-MIAS, où les clichés allaient par paires ; seule la
manière de retrouver la patiente change.


In [ ]:
paths = [image.path for image in trainable]
targets = np.array([CLASS_NAMES.index(image.label) for image in trainable])
patients = [image.patient for image in trainable]

# StratifiedGroupKFold travaille sur des entiers : la clé de cas est encodée,
# l'ordre n'a aucune importance tant que deux clichés d'un même cas partagent
# le même code.
patient_codes = {patient: code for code, patient in enumerate(sorted(set(patients)))}
groups = np.array([patient_codes[patient] for patient in patients])


def group_split(indices: np.ndarray, n_splits: int) -> tuple[np.ndarray, np.ndarray]:
    """Sépare un bloc en (majorité, 1/n_splits), sans jamais couper une patiente."""
    splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    major, minor = next(splitter.split(indices, targets[indices], groups[indices]))
    return indices[major], indices[minor]


all_indices = np.arange(len(trainable))
# 5 plis → ~20 % en test ; le reste est redécoupé pour obtenir ~20 % en validation.
train_val_idx, test_idx = group_split(all_indices, n_splits=3 if SMOKE_TEST else 5)
train_idx, val_idx = group_split(train_val_idx, n_splits=2 if SMOKE_TEST else 4)

splits = {"train": train_idx, "val": val_idx, "test": test_idx}

for name, indices in splits.items():
    positives = int(targets[indices].sum())
    print(
        f"{name:6} {len(indices):5} clichés | {positives:5} malins | "
        f"{len(set(groups[indices])):5} cas"
    )

# Aucune patiente ne doit apparaître dans deux découpages.
for left in splits:
    for right in splits:
        if left < right:
            shared = set(groups[splits[left]]) & set(groups[splits[right]])
            assert not shared, f"fuite {left}/{right} : {len(shared)} cas partagés"
print("\naucun cas partagé entre les découpages")

# Un découpage ne contenant qu'une seule classe rendrait le ROC-AUC indéfini et
# le choix du seuil arbitraire. Mieux vaut le voir ici qu'à l'étape 7.
for name, indices in splits.items():
    present_classes = set(targets[indices].tolist())
    if len(present_classes) < 2:
        print(
            f"⚠️ le découpage {name} ne contient qu'une classe "
            f"({[CLASS_NAMES[c] for c in present_classes]}) : métriques ininterprétables."
        )


## Étape 4 — Les fichiers sont-ils lisibles par le pipeline ?

Le pipeline d'inférence identifie le format sur la **signature du fichier**, pas
sur son extension : `app.ai.preprocessing.loaders.detect_format` reconnaît PNG,
JPEG et DICOM. Contrairement aux PGM de mini-MIAS, qu'il fallait convertir, les
fichiers Mini-DDSM sont déjà dans un format accepté — reste à le vérifier plutôt
qu'à le croire, et à s'assurer qu'aucun fichier n'est tronqué.

Un échantillon est contrôlé ici ; le décodage de la totalité a lieu à l'étape
suivante, où une image illisible fera échouer la construction du jeu de données.


In [ ]:
def check_readable(image: Mammogram) -> tuple[ImageFormat, tuple[int, int]]:
    """Vérifie qu'un fichier est reconnu puis décodé par la chaîne de production."""
    data = image.path.read_bytes()
    image_format = detect_format(data)
    if image_format is None:
        raise ValueError(
            f"{image.path} : signature non reconnue. L'API refuserait ce fichier."
        )
    preprocessed = preprocess_for_inference(data)
    return image_format, preprocessed.original_size


# Un cliché par classe et par incidence : de quoi repérer un dossier partiellement
# corrompu sans relire tout le dataset deux fois.
probes: dict[tuple[str, str], Mammogram] = {}
for image in trainable:
    probes.setdefault((image.label, image.view), image)

formats: Counter = Counter()
for (label, view), image in sorted(probes.items()):
    image_format, original_size = check_readable(image)
    formats[image_format] += 1
    print(
        f"{label:10} {view:3} {image.path.name:28} "
        f"{image_format:5} {original_size[1]}×{original_size[0]}"
    )

assert formats, "aucun cliché à contrôler"
assert set(formats) <= {ImageFormat.JPEG, ImageFormat.PNG}, set(formats)
print("\nformats détectés par le pipeline :", dict(formats))


## Étape 5 — Jeu de données

Chaque image passe par `preprocess_for_inference`, c'est-à-dire **exactement** la
fonction qu'appelle l'API : niveaux de gris → filtre médian → CLAHE →
redimensionnement 384×384 avec remplissage. Rien n'est réimplémenté ici.

Le résultat 8 bits est mis en cache mémoire : le prétraitement d'une image
Mini-DDSM coûte assez cher pour ne pas le refaire à chaque époque. Le coût est
annoncé avant d'être payé — environ 147 Ko par image, soit de l'ordre de 200 Mo
pour le dataset complet, ce qui tient largement dans la RAM d'une instance Colab.

L'augmentation ne s'applique qu'au jeu d'entraînement, et **après** le
prétraitement, sur l'image 8 bits — puis `normalize` termine la chaîne comme à
l'inférence. Deux transformations seulement, géométriques :

- miroir horizontal, qui revient à changer de côté ;
- rotation et zoom légers, qui simulent les variations de positionnement.

Pas de jitter d'intensité : CLAHE vient précisément d'égaliser le contraste, le
perturber ensuite reviendrait à défaire l'étape de prétraitement.


In [ ]:
MAX_ROTATION_DEGREES = 10.0
MAX_ZOOM = 0.10


def augment(image: np.ndarray, rng: random.Random) -> np.ndarray:
    """Miroir et petite transformation affine sur l'image 8 bits prétraitée."""
    if rng.random() < 0.5:
        image = cv2.flip(image, 1)

    angle = rng.uniform(-MAX_ROTATION_DEGREES, MAX_ROTATION_DEGREES)
    scale = 1.0 + rng.uniform(-MAX_ZOOM, MAX_ZOOM)
    height, width = image.shape[:2]
    matrix = cv2.getRotationMatrix2D((width / 2, height / 2), angle, scale)

    # Remplissage noir : c'est déjà la valeur du fond sur une mammographie.
    return cv2.warpAffine(
        image, matrix, (width, height), flags=cv2.INTER_LINEAR, borderValue=0
    )


class MiniDdsmDataset(Dataset):
    """Clichés Mini-DDSM prétraités par la chaîne d'inférence."""

    def __init__(self, name: str, indices: np.ndarray, *, training: bool) -> None:
        self.paths = [paths[i] for i in indices]
        self.targets = [int(targets[i]) for i in indices]
        self.training = training
        self.rng = random.Random(SEED)

        self.images: list[np.ndarray] = []
        for position, path in enumerate(self.paths, start=1):
            try:
                preprocessed = preprocess_for_inference(path.read_bytes())
            except Exception as exc:
                raise ValueError(f"{path} : prétraitement impossible ({exc}).") from exc
            # `display` est l'image 8 bits que le modèle voit, avant normalisation.
            self.images.append(preprocessed.display)
            if position % 200 == 0 or position == len(self.paths):
                print(f"  {name:6} {position:5}/{len(self.paths)} clichés prétraités")

    def __len__(self) -> int:
        return len(self.paths)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, int]:
        image = self.images[index]
        if self.training:
            image = augment(image, self.rng)
        tensor = torch.from_numpy(np.ascontiguousarray(normalize(image)))
        return tensor, self.targets[index]


megabytes = len(trainable) * IMAGE_SIZE[0] * IMAGE_SIZE[1] / 1024**2
print(f"prétraitement de {len(trainable)} clichés (~{megabytes:.0f} Mo en cache)")

datasets = {
    name: MiniDdsmDataset(name, indices, training=(name == "train"))
    for name, indices in splits.items()
}

loaders = {
    name: DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=(name == "train"),
        num_workers=0,  # Colab : les workers coûtent plus qu'ils ne rapportent ici.
        drop_last=False,
    )
    for name, dataset in datasets.items()
}

batch, batch_targets = next(iter(loaders["train"]))
print("\nlot d'entraînement :", tuple(batch.shape), batch.dtype)
print("cibles             :", batch_targets.tolist())
assert batch.shape[1:] == (3, *IMAGE_SIZE), batch.shape


## Étape 6 — Modèle et entraînement

Le réseau est construit par `app.ai.inference.loader.build_model` : la même
fabrique que celle du chargement, donc une architecture nécessairement
compatible avec le checkpoint produit.

Deux taux d'apprentissage : la tête part de zéro et apprend vite, le corps
pré-entraîné se contente d'un ajustement fin. Mini-DDSM reste très en deçà de ce
qui permettrait de réapprendre un corps de réseau entier.

La perte est pondérée par l'inverse de la fréquence des classes. Le
déséquilibre bénin / malin est faible sur Mini-DDSM — les deux dossiers ont des
effectifs voisins — mais la pondération se calcule sur les effectifs réels du
découpage d'entraînement, pas sur ceux du dataset entier, et coûte peu si elle
vaut 1.

Le meilleur modèle est retenu sur le **rappel malin** en validation, pas sur
l'accuracy : manquer un cancer est la défaillance la plus grave du système. Le
ROC-AUC sert de départage. Le rappel seul se maximise trivialement en répondant
« malin » à tout : un tel modèle atteindrait 1,00 dès la première époque et
resterait sélectionné jusqu'au bout sans rien discriminer. Ce risque ne
disparaît pas avec dix fois plus de données — il devient seulement moins
probable.


In [ ]:
model = build_model(ARCHITECTURE, pretrained=True).to(DEVICE)

# La tête (créée par build_model) apprend vite ; le corps ImageNet est ajusté
# doucement.
head_parameters = list(model.classifier.parameters())
head_ids = {id(p) for p in head_parameters}
backbone_parameters = [p for p in model.parameters() if id(p) not in head_ids]

optimizer = torch.optim.AdamW(
    [
        {"params": backbone_parameters, "lr": LR_BACKBONE},
        {"params": head_parameters, "lr": LR_HEAD},
    ],
    weight_decay=WEIGHT_DECAY,
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

train_targets = targets[splits["train"]]
frequencies = np.bincount(train_targets, minlength=len(CLASS_NAMES)).astype(np.float64)
weights = frequencies.sum() / (len(CLASS_NAMES) * np.maximum(frequencies, 1.0))
criterion = nn.CrossEntropyLoss(
    weight=torch.tensor(weights, dtype=torch.float32, device=DEVICE)
)

print("effectifs d'entraînement :", dict(zip(CLASS_NAMES, frequencies.astype(int))))
print("pondération de la perte  :", dict(zip(CLASS_NAMES, weights.round(3))))


In [ ]:
@torch.no_grad()
def evaluate(loader: DataLoader) -> tuple[np.ndarray, np.ndarray]:
    """Retourne (cibles, probabilités de malignité) pour un jeu complet."""
    model.eval()
    all_targets: list[int] = []
    all_scores: list[float] = []

    for inputs, batch_targets in loader:
        probabilities = torch.softmax(model(inputs.to(DEVICE)), dim=1)
        all_scores.extend(probabilities[:, MALIGNANT_INDEX].cpu().tolist())
        all_targets.extend(batch_targets.tolist())

    return np.array(all_targets), np.array(all_scores)


def malignant_recall(y_true: np.ndarray, scores: np.ndarray, threshold: float) -> float:
    predictions = (scores >= threshold).astype(int)
    return float(recall_score(y_true, predictions, pos_label=1, zero_division=0))


best_score = (-1.0, -1.0)
best_state: dict[str, torch.Tensor] | None = None
best_epoch = 0
history: list[dict[str, float]] = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0

    for inputs, batch_targets in loaders["train"]:
        inputs = inputs.to(DEVICE)
        batch_targets = batch_targets.to(DEVICE)

        optimizer.zero_grad()
        loss = criterion(model(inputs), batch_targets)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)

    scheduler.step()

    train_loss = running_loss / len(datasets["train"])
    val_targets, val_scores = evaluate(loaders["val"])
    recall = malignant_recall(val_targets, val_scores, 0.5)
    # ROC-AUC indéfini si la validation ne contient qu'une seule classe.
    auc = (
        float(roc_auc_score(val_targets, val_scores))
        if len(set(val_targets.tolist())) > 1
        else float("nan")
    )

    history.append(
        {"epoch": epoch, "train_loss": train_loss, "val_recall": recall, "val_auc": auc}
    )
    print(
        f"époque {epoch:3} | perte {train_loss:.4f} | "
        f"rappel malin (val) {recall:.3f} | ROC-AUC (val) {auc:.3f}"
    )

    # Le rappel malin décide, le ROC-AUC départage. Sans ce second critère, un
    # modèle qui répondrait « malin » à tout obtiendrait un rappel de 1,0 dès la
    # première époque et resterait sélectionné jusqu'à la fin, alors qu'il ne
    # discrimine rien.
    score = (recall, 0.0 if np.isnan(auc) else auc)
    if score > best_score:
        best_score = score
        best_epoch = epoch
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    if epoch - best_epoch >= EARLY_STOPPING_PATIENCE:
        print(f"arrêt anticipé : aucun progrès depuis l'époque {best_epoch}")
        break

assert best_state is not None, "aucune époque n'a été exécutée"
model.load_state_dict(best_state)
print(
    f"\nmeilleur état retenu : époque {best_epoch} "
    f"(rappel malin {best_score[0]:.3f}, ROC-AUC {best_score[1]:.3f})"
)

#: ROC-AUC de validation de l'époque retenue. Sert de témoin : le modèle évalué
#: puis sauvegardé plus bas doit reproduire cette valeur, sinon ce ne sont plus
#: les mêmes poids.
BEST_VAL_AUC = float(history[best_epoch - 1]["val_auc"])


def restore_best_state() -> None:
    """Réapplique au modèle en mémoire les poids de l'époque retenue.

    Appelée en tête de chaque cellule qui évalue ou sauvegarde, ce qui rend la
    fin du notebook indifférente à l'ordre d'exécution des cellules.

    Sans elle, ré-exécuter la cellule précédente — celle qui fait
    `build_model(ARCHITECTURE, pretrained=True)` — remplace le modèle entraîné
    par un réseau ImageNet neuf, et tout ce qui suit porte sur ce réseau : les
    métriques de test, le checkpoint écrit, le modèle déployé. C'est arrivé, et
    rien ne l'a signalé : les métriques de validation, calculées avant la
    ré-exécution, restaient bonnes dans la fiche du modèle.
    """
    assert best_state is not None, "aucune époque sélectionnée : relancer l'entraînement"
    model.load_state_dict(best_state)


## Étape 7 — Choix du seuil

Le seuil n'est pas laissé à 0,5 : il est choisi sur la validation comme le seuil
**le plus haut** qui atteint encore la sensibilité visée. À sensibilité donnée,
c'est celui qui produit le moins de faux positifs.

Si aucun seuil n'atteint la cible, on retombe sur l'indice de Youden, et le
notebook le signale.


In [ ]:
def choose_threshold(
    y_true: np.ndarray, scores: np.ndarray, target_sensitivity: float
) -> tuple[float, str]:
    """Seuil le plus élevé atteignant la sensibilité cible, sinon Youden."""
    candidates = np.unique(np.concatenate([scores, [0.0, 1.0]]))

    achieving = [
        threshold
        for threshold in candidates
        if malignant_recall(y_true, scores, float(threshold)) >= target_sensitivity
    ]
    if achieving:
        threshold = float(max(achieving))
        return threshold, (
            f"seuil le plus élevé atteignant une sensibilité ≥ {target_sensitivity:.2f} "
            "sur la validation"
        )

    def youden(threshold: float) -> float:
        predictions = (scores >= threshold).astype(int)
        matrix = confusion_matrix(y_true, predictions, labels=[0, 1])
        true_negative, false_positive, false_negative, true_positive = matrix.ravel()
        sensitivity = true_positive / max(true_positive + false_negative, 1)
        specificity = true_negative / max(true_negative + false_positive, 1)
        return sensitivity + specificity - 1

    threshold = float(max(candidates, key=lambda t: youden(float(t))))
    return threshold, (
        f"sensibilité de {target_sensitivity:.2f} inatteignable sur la validation : "
        "seuil retenu par l'indice de Youden"
    )


# Les poids évalués sont ceux de l'époque retenue, quoi qu'il ait été
# ré-exécuté entre-temps.
restore_best_state()

val_targets, val_scores = evaluate(loaders["val"])
threshold, threshold_rationale = choose_threshold(
    val_targets, val_scores, TARGET_SENSITIVITY
)

print(f"seuil retenu : {threshold:.4f}")
print(f"raison       : {threshold_rationale}")
print(f"sensibilité en validation : {malignant_recall(val_targets, val_scores, threshold):.3f}")


## Étape 8 — Évaluation sur le jeu de test

Le jeu de test compte quelques centaines de clichés, contre une vingtaine pour
mini-MIAS : les chiffres deviennent lisibles. Ils restent mesurés **sur le même
corpus que l'entraînement** — mêmes appareils, mêmes années, même population.
Une validation externe reste à faire, et rien ici n'en tient lieu.


In [ ]:
def compute_metrics(y_true: np.ndarray, scores: np.ndarray, threshold: float) -> dict:
    predictions = (scores >= threshold).astype(int)
    matrix = confusion_matrix(y_true, predictions, labels=[0, 1])
    true_negative, false_positive, false_negative, true_positive = (
        int(value) for value in matrix.ravel()
    )

    return {
        "n_images": int(len(y_true)),
        "accuracy": float((predictions == y_true).mean()),
        "precision": float(precision_score(y_true, predictions, zero_division=0)),
        "recall_malignant": float(recall_score(y_true, predictions, zero_division=0)),
        "specificity": float(true_negative / max(true_negative + false_positive, 1)),
        "f1": float(f1_score(y_true, predictions, zero_division=0)),
        "roc_auc": (
            float(roc_auc_score(y_true, scores))
            if len(set(y_true.tolist())) > 1
            else None
        ),
        "confusion_matrix": {
            "true_negative": true_negative,
            "false_positive": false_positive,
            "false_negative": false_negative,
            "true_positive": true_positive,
        },
    }


restore_best_state()

test_targets, test_scores = evaluate(loaders["test"])
test_metrics = compute_metrics(test_targets, test_scores, threshold)
val_metrics = compute_metrics(val_targets, val_scores, threshold)

for key, value in test_metrics.items():
    if key != "confusion_matrix":
        print(f"{key:18} {value}")

matrix = test_metrics["confusion_matrix"]
print("\n                 prédit bénin  prédit malin")
print(f"réel bénin       {matrix['true_negative']:12}  {matrix['false_positive']:12}")
print(f"réel malin       {matrix['false_negative']:12}  {matrix['true_positive']:12}")

if matrix["false_negative"]:
    print(
        f"\n⚠️ {matrix['false_negative']} cancer(s) manqué(s) sur "
        f"{matrix['false_negative'] + matrix['true_positive']} — "
        "c'est la défaillance la plus grave du système."
    )

# Un modèle qui répond la même chose à tout obtient d'excellentes métriques sur
# une des deux classes. Mieux vaut que le notebook le dise que de laisser lire
# un rappel de 1,00 comme une réussite.
#
# Tolérance, et non égalité stricte à zéro : lors de l'incident du 15/08 la
# spécificité valait 0,0019 — un cliché bénin sur 536 tombé du bon côté — et
# `== 0.0` n'a rien déclenché sur un modèle qui répondait pourtant « malin » à
# tout. Un flottant issu d'un comptage ne vaut presque jamais exactement zéro.
DEGENERATE_TOLERANCE = 0.01

is_degenerate = (
    test_metrics["specificity"] <= DEGENERATE_TOLERANCE
    or test_metrics["recall_malignant"] <= DEGENERATE_TOLERANCE
)
if is_degenerate:
    print(
        "\n⛔ MODÈLE DÉGÉNÉRÉ : toutes les images ou presque reçoivent la même "
        "classe au seuil retenu. Ce checkpoint ne discrimine rien et ne doit pas "
        "être déployé, quelles que soient les autres métriques. L'étape 9 "
        "refusera de l'écrire."
    )


## Étape 9 — Enregistrement du checkpoint

Le format est celui qu'attend `app.ai.inference.loader` :

- `class_names` repris de `app.ai.CLASS_NAMES`, dans l'ordre figé ;
- `preprocessing_version` repris de `app.ai.preprocessing` ;
- `version` **sans** le préfixe `placeholder-`, réservé aux modèles de
  substitution et refusé par le chargeur pour un modèle entraîné ;
- `clinically_validated` à **`False`**, explicitement. Dix fois plus de données
  que mini-MIAS ne change rien à ce drapeau : il ne mesure pas la taille du jeu
  d'entraînement mais l'existence d'une revue clinique humaine documentée, qui
  n'a pas eu lieu. Aucune cellule de ce notebook n'autorise à le passer à `True`.

Les clés supplémentaires sont ignorées par le chargeur mais lues par les
humains : elles évitent qu'un `.pt` retrouvé dans six mois soit un fichier de
poids anonyme. Une fiche modèle JSON les reprend, comme le demande
`models/README.md`.

### Trois contrôles avant d'écrire

Le 15 août 2026, ce notebook a produit un checkpoint contenant un réseau
ImageNet intact, sous des métadonnées annonçant treize époques et une époque
sélectionnée. La cellule de l'étape 6 avait été ré-exécutée après
l'entraînement : `model` était redevenu un `build_model(pretrained=True)` neuf,
et `torch.save` a écrit cet état. Le fichier a passé toutes les validations de
contrat et a été déployé, où il répondait 50 % à toute image.

Trois contrôles ferment cette porte, et **aucun checkpoint n'est écrit s'ils
échouent** :

| Contrôle | Ce qu'il attrape |
|----------|------------------|
| Corps du réseau ≠ ImageNet | un modèle jamais entraîné, quoi qu'en disent ses métadonnées |
| ROC-AUC recalculé ≈ celui de l'époque retenue | des poids qui ne sont plus ceux qui ont été sélectionnés |
| Modèle non dégénéré | un réseau qui répond la même classe à tout |

S'y ajoute un changement de fond : ce qui est sauvegardé est `best_state`, l'état
sélectionné à l'étape 6, et non `model.state_dict()` — l'état que le modèle en
mémoire se trouve avoir. Les cellules d'évaluation appellent par ailleurs
`restore_best_state()`, ce qui rend toute la fin du notebook indifférente à
l'ordre d'exécution des cellules.


In [ ]:
def git_commit(repo: Path) -> str | None:
    try:
        result = subprocess.run(
            ["git", "-C", str(repo), "rev-parse", "--short", "HEAD"],
            capture_output=True,
            text=True,
            check=True,
        )
    except (OSError, subprocess.CalledProcessError):
        return None
    return result.stdout.strip()


assert not MODEL_VERSION.startswith("placeholder-"), (
    "le préfixe placeholder- est réservé aux modèles de substitution"
)

# --------------------------------------------------------------------------- #
# Contrôles d'intégrité — rien n'est écrit tant qu'ils ne sont pas passés.
#
# Les métadonnées de cette cellule décrivent ce que le notebook *croit*
# sauvegarder. Elles ne prouvent rien sur les poids réellement écrits : un
# checkpoint annonçant treize époques et une époque retenue a déjà été produit
# à partir d'un réseau ImageNet intact, et il a été déployé.
# --------------------------------------------------------------------------- #

#: Écart maximal toléré entre le ROC-AUC recalculé avant sauvegarde et celui de
#: l'époque sélectionnée. L'évaluation étant déterministe et la validation sans
#: augmentation, les deux valeurs devraient être identiques ; la marge ne couvre
#: que le non-déterminisme des noyaux CUDA.
SELECTION_AUC_TOLERANCE = 0.01

restore_best_state()


def backbone_matches_imagenet(state_dict: dict[str, torch.Tensor]) -> bool:
    """Le corps du réseau est-il resté exactement celui d'ImageNet ?

    Un seul pas de descente de gradient modifie tous les poids du corps. Un
    corps identique bit à bit signifie donc qu'aucun entraînement n'a eu lieu,
    quelles que soient les métriques calculées plus haut. La tête est exclue :
    `build_model` la remplace à chaque construction, elle diffère de la
    référence même sur un réseau vierge.
    """
    reference = build_model(ARCHITECTURE, pretrained=True).state_dict()
    for name, tensor in reference.items():
        if name.startswith("classifier."):
            continue
        saved = state_dict.get(name)
        if saved is None or not torch.equal(saved.cpu(), tensor.cpu()):
            return False
    return True


if backbone_matches_imagenet(best_state):
    raise ValueError(
        "le corps du réseau est identique aux poids ImageNet : l'entraînement "
        "n'y a laissé aucune trace, ce checkpoint répondrait 50 % à toute "
        "image. Cause habituelle : la cellule de l'étape 6 qui construit le "
        "modèle a été ré-exécutée après l'entraînement. Relancer le notebook de "
        "haut en bas, sans ré-exécution partielle."
    )

# Le modèle sur le point d'être écrit doit reproduire la mesure qui l'a fait
# sélectionner. Sinon, ce ne sont plus les poids de l'époque retenue.
selection_targets, selection_scores = evaluate(loaders["val"])
if not np.isnan(BEST_VAL_AUC) and len(set(selection_targets.tolist())) > 1:
    selection_auc = float(roc_auc_score(selection_targets, selection_scores))
    if abs(selection_auc - BEST_VAL_AUC) > SELECTION_AUC_TOLERANCE:
        raise ValueError(
            f"ROC-AUC de validation {selection_auc:.4f} au moment d'écrire, contre "
            f"{BEST_VAL_AUC:.4f} à l'époque {best_epoch} qui a été sélectionnée. "
            "Les poids en mémoire ne sont pas ceux qui ont été choisis : ne rien "
            "écrire tant que l'écart n'est pas expliqué."
        )
    print(
        f"contrôle de sélection : ROC-AUC {selection_auc:.4f} "
        f"≈ {BEST_VAL_AUC:.4f} (époque {best_epoch})"
    )

if is_degenerate:
    if SMOKE_TEST:
        # Une époque sur une poignée d'images produit presque toujours un modèle
        # dégénéré : bloquer ici empêcherait de vérifier la mécanique du
        # notebook, ce qui est précisément l'objet du mode vérification.
        print(
            "mode vérification : modèle dégénéré toléré, le checkpoint produit "
            "n'a aucune vocation à être déployé."
        )
    else:
        raise ValueError(
            "modèle dégénéré (voir l'étape 8) : toutes les images ou presque "
            "reçoivent la même classe. Un tel checkpoint ne doit pas atteindre "
            "models/, où il serait indiscernable d'un modèle utile."
        )

LIMITATIONS = [
    f"Entraîné sur {len(trainable)} clichés Mini-DDSM (Benign + Cancer) ; "
    f"{normal_count} clichés Normal écartés faute de classe `normal` dans le "
    "contrat de sortie.",
    "Étiquetage asymétrique : la classe maligne est restreinte aux clichés "
    f"portant un masque de lésion ({mask_report['malignant_dropped']} clichés de "
    f"cas Cancer écartés, {mask_report['cases_lost']} cas perdus), tandis que la "
    "classe bénigne garde toutes les vues du cas. `benign` reste juste sur un "
    "sein sain — bénin et normal disent tous deux « pas de cancer » — alors que "
    "`malignant` sur un sein sain apprendrait une association fausse.",
    "La restriction dépend du nommage des masques (`…_Mask.jpg`) : un masque non "
    "reconnu retire son cliché de la classe maligne. Le rapport de l'étape 1 "
    "chiffre les fichiers non appariés.",
    "Mini-DDSM dérive de films numérisés (DDSM, années 1990-2000), recompressés "
    "en JPEG 8 bits : ni la dynamique d'origine ni les mammographes numériques "
    "actuels n'y sont représentés.",
    f"Métriques mesurées sur {len(splits['test'])} clichés du même corpus que "
    "l'entraînement. Aucune validation sur une population externe, ni par tranche "
    "d'âge, densité mammaire ou constructeur.",
    "Aucune revue clinique : `clinically_validated` reste à False.",
]

metadata = {
    "architecture": ARCHITECTURE,
    "class_names": list(CLASS_NAMES),
    "preprocessing_version": PREPROCESSING_VERSION,
    "threshold": float(threshold),
    "version": MODEL_VERSION,
    # Explicite, alors que le chargeur retomberait de toute façon sur False en
    # l'absence de clé : ce modèle est entraîné mais n'a fait l'objet d'aucune
    # revue clinique. Le passer à True demande une validation humaine documentée
    # menée hors de ce notebook — voir DEFAULT_CLINICALLY_VALIDATED dans
    # app/ai/inference/loader.py. Aucun résultat de cette cellule ne l'autorise,
    # et la taille du jeu d'entraînement n'entre pas dans cette décision.
    "clinically_validated": False,
    "image_size": list(IMAGE_SIZE),
    "dataset": {
        "name": DATASET_NAME,
        "source": DATASET_SOURCE,
        "images_total": len(images),
        "images_normal_excluded": normal_count,
        "images_used": len(trainable),
        "patients_used": len(set(patients)),
        "class_counts": counts,
        "label_source": "dossier de premier niveau (Benign/Cancer/Normal)",
        "label_policy": {
            "benign": "toutes les vues du cas",
            "malignant": "clichés portant un masque de lésion, uniquement",
            "normal": "exclu (contrat de sortie à deux classes)",
            "rationale": (
                "`malignant` sur un sein sain apprend une association fausse dans "
                "le sens le plus dangereux ; `benign` sur un sein sain reste juste "
                "pour le triage, bénin et normal signifiant tous deux « pas de "
                "cancer »."
            ),
            **mask_report,
        },
        "masks_indexed": sum(masks.values()),
        "files_skipped": dict(skipped),
    },
    "split": {
        "strategy": "StratifiedGroupKFold par cas DDSM (préfixe + numéro de cas)",
        "seed": SEED,
        **{name: int(len(indices)) for name, indices in splits.items()},
        "patients": {
            name: len(set(groups[indices])) for name, indices in splits.items()
        },
    },
    "training": {
        "epochs_run": len(history),
        "best_epoch": best_epoch,
        "batch_size": BATCH_SIZE,
        "lr_head": LR_HEAD,
        "lr_backbone": LR_BACKBONE,
        "weight_decay": WEIGHT_DECAY,
        "class_weights": dict(zip(CLASS_NAMES, weights.tolist())),
        "selection_criterion": (
            "rappel sur la classe maligne en validation, ROC-AUC en départage"
        ),
        "augmentation": "miroir horizontal, rotation ±10°, zoom ±10 %",
        "smoke_test": SMOKE_TEST,
    },
    "threshold_rationale": threshold_rationale,
    "metrics": {"validation": val_metrics, "test": test_metrics},
    "limitations": LIMITATIONS,
    "trained_at": datetime.now(timezone.utc).isoformat(),
    "seed": SEED,
    "git_commit": git_commit(REPO_ROOT),
}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# Le nom de fichier reprend la version du modèle : un .pt et sa fiche restent
# rattachables à ce qu'annonce le checkpoint.
label = MODEL_VERSION.removeprefix(f"{ARCHITECTURE}-")
stem = f"breastai_{ARCHITECTURE}_{label}_{datetime.now().strftime('%Y%m%d')}"
checkpoint_path = OUTPUT_DIR / f"{stem}.pt"
model_card_path = OUTPUT_DIR / f"{stem}.json"

# `best_state` et non `model.state_dict()` : ce qui est écrit est exactement
# l'état sélectionné à l'étape 6, pas ce que le modèle en mémoire se trouve
# contenir au moment où cette cellule s'exécute.
torch.save({**metadata, "state_dict": best_state}, checkpoint_path)
model_card_path.write_text(json.dumps(metadata, indent=2, ensure_ascii=False), "utf-8")

print(f"checkpoint  : {checkpoint_path}")
print(f"fiche modèle: {model_card_path}")


## Étape 10 — Relire le checkpoint avec le code de production

Le notebook ne se termine pas sur un `torch.save`. Le fichier est rechargé par
**`load_checkpoint`**, celui-là même qu'exécute l'API au démarrage, puis une
prédiction est produite via `Predictor`. Un contrat cassé échoue ici, sur le
poste qui a entraîné le modèle, et non au démarrage du serveur.

S'y ajoute une comparaison de non-régression : sur un échantillon témoin de la
validation, le modèle rechargé depuis le disque doit produire les mêmes scores
que celui resté en mémoire. Un écart de ROC-AUC supérieur à 0,02 lève une
erreur.

Ce contrôle couvre la **sérialisation** — poids manquants, tampons de BatchNorm
perdus, chargement partiel. Il ne remplace pas ceux de l'étape 9, et il faut
être clair sur sa portée : lors de l'incident du 15 août, mémoire et disque
étaient parfaitement d'accord, tous deux sur un réseau ImageNet. Un checkpoint
peut être fidèlement écrit *et* faux. C'est l'étape 9 qui attrape ce cas-là.


In [ ]:
bundle = load_checkpoint(checkpoint_path, torch.device("cpu"))

assert bundle.is_placeholder is False, "le modèle ne doit pas être vu comme placeholder"
assert bundle.class_names == CLASS_NAMES
assert bundle.preprocessing_version == PREPROCESSING_VERSION
assert bundle.architecture == ARCHITECTURE
assert abs(bundle.threshold - threshold) < 1e-9
# Ne plus être un placeholder ne vaut pas validation clinique : c'est
# précisément la confusion que le drapeau sépare.
assert bundle.clinically_validated is False

print(f"version           : {bundle.version}")
print(f"architecture      : {bundle.architecture}")
print(f"classes           : {bundle.class_names}")
print(f"seuil             : {bundle.threshold:.4f}")
print(f"prétraitement     : {bundle.preprocessing_version}")
print(f"placeholder       : {bundle.is_placeholder}")
print(f"validé cliniquement : {bundle.clinically_validated}")

# --------------------------------------------------------------------------- #
# Non-régression : le modèle rechargé doit répondre comme celui en mémoire.
#
# Ce contrôle couvre la sérialisation — poids manquants, tampons de BatchNorm
# perdus, chargement partiel. Il ne remplace pas ceux de l'étape 9 : lors de
# l'incident du 15/08, mémoire et disque étaient d'accord, tous deux sur un
# réseau ImageNet. Un checkpoint peut être fidèlement écrit *et* faux.
# --------------------------------------------------------------------------- #
RELOAD_AUC_TOLERANCE = 0.02
PROBE_SIZE = 64

probe_subset = torch.utils.data.Subset(
    datasets["val"], list(range(min(PROBE_SIZE, len(datasets["val"]))))
)
probe_loader = DataLoader(probe_subset, batch_size=BATCH_SIZE, shuffle=False)


@torch.no_grad()
def scores_of(network: nn.Module, device: torch.device) -> tuple[np.ndarray, np.ndarray]:
    """Cibles et probabilités de malignité d'un réseau sur l'échantillon témoin."""
    network.eval()
    seen_targets: list[int] = []
    seen_scores: list[float] = []
    for inputs, batch_targets in probe_loader:
        probabilities = torch.softmax(network(inputs.to(device)), dim=1)
        seen_scores.extend(probabilities[:, MALIGNANT_INDEX].cpu().tolist())
        seen_targets.extend(batch_targets.tolist())
    return np.array(seen_targets), np.array(seen_scores)


restore_best_state()
memory_targets, memory_scores = scores_of(model, DEVICE)
reloaded_targets, reloaded_scores = scores_of(bundle.model, bundle.device)

assert (memory_targets == reloaded_targets).all(), "échantillon témoin désaligné"

largest_gap = float(np.max(np.abs(memory_scores - reloaded_scores)))
print(f"\nécart maximal de score sur {len(memory_scores)} images : {largest_gap:.6f}")

if len(set(memory_targets.tolist())) > 1:
    memory_auc = float(roc_auc_score(memory_targets, memory_scores))
    reloaded_auc = float(roc_auc_score(reloaded_targets, reloaded_scores))
    print(f"ROC-AUC mémoire {memory_auc:.4f} | rechargé {reloaded_auc:.4f}")
    if abs(memory_auc - reloaded_auc) > RELOAD_AUC_TOLERANCE:
        raise ValueError(
            f"ROC-AUC {memory_auc:.4f} en mémoire contre {reloaded_auc:.4f} après "
            "rechargement du checkpoint qui vient d'être écrit. Le fichier ne "
            "contient pas le modèle évalué : ne pas le déployer."
        )
else:
    print("échantillon témoin monoclasse : comparaison sur les scores seuls.")


# Une prédiction complète, du fichier déposé au résultat rendu par l'API.
probe_index = int(splits["test"][0])
probe_image = trainable[probe_index]
probe = preprocess_for_inference(probe_image.path.read_bytes())
result = Predictor(bundle).predict(probe.tensor)

print(f"\ncliché {probe_image.path.name} (cas {probe_image.patient}, "
      f"vérité terrain : {probe_image.label})")
print(f"  prédiction  : {result.label}")
print(f"  p(malin)    : {result.probability:.4f}")
print(f"  confiance   : {result.confidence:.4f}")
print(f"  version     : {result.model_version}")

# Grad-CAM tourne sur la dernière couche convolutive : elle doit exister.
assert bundle.target_layer is not None
print("\ncontrat de checkpoint vérifié de bout en bout")


## Déployer, et après

```bash
cp models/breastai_efficientnet_b0_miniddsm-v1_<date>.pt models/breastai_efficientnet.pt
```

`MODEL_PATH` pointe dessus par défaut. Aucun autre changement n'est nécessaire :
ni l'inférence, ni les services, ni l'API ne bougent. Les analyses déjà rendues
par les modèles précédents restent identifiables par leur `model_version` —
`placeholder-…` pour le placeholder, `efficientnet_b0-mini-mias-v1` pour le
modèle intérimaire — et peuvent être rejouées par `POST /analyses/{id}/infer`.

`clinically_validated` reste `false` : l'interface et les rapports continuent
d'afficher le bandeau « modèle entraîné mais non validé cliniquement ». Il ne
disparaîtra qu'au terme d'une revue humaine documentée, qui ne se décide pas
dans un notebook.

### Ce que ce modèle n'est pas

Il a vu quelques centaines à un millier de clichés issus d'un unique corpus de
films numérisés, recompressés en 8 bits. Il n'a jamais vu de cliché normal,
puisque le contrat de sortie ne comporte pas cette classe : présenté à un sein
sain, il répondra `benign` ou `malignant`, sans troisième possibilité — et
`benign` est alors la réponse attendue, c'est le sens qu'a cette sortie ici.

Sa classe maligne ne contient que des clichés porteurs d'une lésion annotée ;
sa classe bénigne mêle des seins porteurs de lésions bénignes et des seins
sains du même cas. C'est un choix documenté à l'étape 2 bis, pas un oubli.

**L'avertissement clinique reste inchangé** : aucune décision médicale ne doit
s'appuyer sur ces prédictions.
